![Logo SENATI](logo_senati.png)

**SENATI | Escuela de Tecnologías de la Información**  
**Taller: Machine Learning y Deep Learning**  
*Instructor: Ing. Gunar Castro Iñigo*

---

# Sistema de estimación de pedidos de kits de robótica — caso práctico intermedio

**Contexto:** Una tienda educativa quiere anticipar pedidos semanales. El número de consultas y la inversión publicitaria pueden asociarse con más ventas; el indicador de campaña escolar toma 0 o 1.

**Objetivo:** aplicar regresión múltiple desde lectura de datos hasta una web pequeña.

## Paso 1. Explorar el CSV
Buscamos las columnas de entrada `X`: `consultas_web`, `inversion_publicidad_soles`, `campana_escolar`, y la respuesta `y`: `pedidos_kits`.

**¿Qué podemos preguntar al grupo?** ¿Qué significa cada variable en la vida real? ¿Cuál sería el peligro de usar esta predicción sin verificarla?

In [1]:
import pandas as pd

datos = pd.read_csv('datos.csv', encoding='utf-8-sig')  # utf-8-sig evita el carácter oculto (BOM) del archivo
print(datos.head())
print()
print('Filas y columnas:', datos.shape)
print()
print('Valores ausentes por columna:')
print(datos.isna().sum())
print()
print(datos.describe().round(2))

  semana  consultas_web  ...  campana_escolar  pedidos_kits
0   S001             31  ...                1            22
1   S002             51  ...                0            20
2   S003             27  ...                0            17
3   S004             45  ...                0            21
4   S005             40  ...                1            16

[5 rows x 5 columns]

Filas y columnas: (110, 5)

Valores ausentes por columna:
semana                        0
consultas_web                 0
inversion_publicidad_soles    0
campana_escolar               0
pedidos_kits                  0
dtype: int64

       consultas_web  inversion_publicidad_soles  campana_escolar  pedidos_kits
count         110.00                      110.00           110.00        110.00
mean           32.22                      577.73             0.51         19.36
std            18.69                      323.26             0.50          7.73
min             5.00                        0.00             0.00

**Significado de cada variable**

- **consultas_web**: cantidad de consultas que recibe la tienda en la semana (interés de los clientes).
- **inversion_publicidad_soles**: dinero gastado en publicidad esa semana, en soles.
- **campana_escolar**: 1 si hubo campaña en colegios, 0 si no (variable binaria).
- **pedidos_kits** (y): cantidad de kits de robótica pedidos en la semana, lo que queremos predecir.

**Revisión:** el archivo tiene 110 filas y no tiene valores ausentes, por lo que no hace falta limpiar ni imputar datos. ¿Peligro de usar la predicción sin verificarla? Comprar o producir kits de más o de menos: si el modelo falla, se pierde dinero en inventario o se pierden ventas. Además los datos son simulados, así que no representan a una tienda real.

## Paso 2. Entrenar
Usamos las 3 variables como entrada `X`. Reservamos el **20 %** de las filas para prueba y entrenamos con el **80 %**. No se usan filas de prueba para entrenar.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

columnas = ['consultas_web', 'inversion_publicidad_soles', 'campana_escolar']
objetivo = 'pedidos_kits'
X = datos[columnas]
y = datos[objetivo]

# Separar 80 % entrenamiento / 20 % prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Filas de entrenamiento:', len(X_train))
print('Filas de prueba:', len(X_test))

# Entrenar
modelo = LinearRegression()
modelo.fit(X_train, y_train)

# Coeficientes
print()
print('Intercepto:', round(modelo.intercept_, 4))
for nombre, coef in zip(columnas, modelo.coef_):
    print(f'  {nombre}: {coef:.4f}')

# Predecir y evaluar con los datos de prueba
pred = modelo.predict(X_test)
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)
print()
print(f'MAE = {mae:.2f}')
print(f'R2  = {r2:.4f}')

Filas de entrenamiento: 88
Filas de prueba: 22

Intercepto: 1.5385
  consultas_web: 0.3463
  inversion_publicidad_soles: 0.0067
  campana_escolar: 5.0182

MAE = 2.95
R2  = 0.7911


### Interpretación de los coeficientes
La ecuación es: `pedidos_kits = intercepto + b1·consultas_web + b2·inversion_publicidad_soles + b3·campana_escolar`

- **consultas_web (0.3463)**: por cada consulta web adicional se esperan ≈ 0.35 pedidos más (10 consultas más ≈ 3.5 pedidos), manteniendo lo demás igual.
- **inversion_publicidad_soles (0.0067)**: cada S/ 100 extra en publicidad suma ≈ 0.67 pedidos; el efecto es pequeño por sol invertido.
- **campana_escolar (5.0182)**: una semana con campaña escolar trae ≈ 5.0 pedidos más que una sin campaña.
- **Intercepto (1.5385)**: valor base cuando todo vale 0 (solo es un punto de partida de la recta, no un caso real).

### Interpretación de MAE y R²
- **Filas reservadas:** 22 de 110 (20 %) para prueba; 88 para entrenar.
- **MAE = 2.95 pedidos:** en promedio, la predicción se equivoca por ≈ 2.95 pedidos. Como el promedio real es ≈ 19.4 pedidos, el error típico es ≈ 15.3 % del valor medio.
- **R² = 0.791:** el modelo explica ≈ 79.1 % de la variación de `pedidos_kits` en los datos de prueba; el resto (20.9 %) depende de factores que no están en el modelo o de ruido.

In [3]:
import json

# Guardar los coeficientes para usarlos en la app Flask
info = {
    'columnas': columnas,
    'intercepto': float(modelo.intercept_),
    'coeficientes': [float(v) for v in modelo.coef_],
    'metricas': {'mae': round(float(mae), 2), 'r2': round(float(r2), 4)},
}
with open('modelo.json', 'w', encoding='utf-8') as f:
    json.dump(info, f, indent=2, ensure_ascii=False)
print(open('modelo.json', encoding='utf-8').read())

{
  "columnas": [
    "consultas_web",
    "inversion_publicidad_soles",
    "campana_escolar"
  ],
  "intercepto": 1.5384729585592574,
  "coeficientes": [
    0.3463033562740661,
    0.006699109182241925,
    5.018213367145449
  ],
  "metricas": {
    "mae": 2.95,
    "r2": 0.7911
  }
}


## Paso 3. Aplicación web (Flask + HTML + CSS, sin JavaScript)
Estructura del proyecto:

```
caso01_kits_robotica/
├── app.py
├── entrenar.py
├── modelo.json
├── datos.csv
├── templates/
│   └── index.html
└── static/
    ├── estilos.css
    └── logo_senati.png
```

La página tiene título, 3 campos con unidades, botón **Calcular**, resultado y una advertencia de estimación. Valida los rangos antes de predecir.

**app.py**
```python
# app.py — Aplicación Flask: Estimador de pedidos semanales de kits de robótica
# Solo Python + Flask + HTML + CSS (sin JavaScript).
import json
from flask import Flask, render_template, request

app = Flask(__name__)

# 1) Cargamos los coeficientes guardados por entrenar.py
with open("modelo.json", encoding="utf-8") as f:
    MODELO = json.load(f)

# 2) Campos del formulario (nombre, etiqueta, unidad y rango permitido)
CAMPOS = [
    {'n': 'consultas_web', 'et': 'Consultas web de la semana', 'u': 'consultas', 'mn': 5, 'mx': 70, 'step': 1, 'tipo': 'num'},
    {'n': 'inversion_publicidad_soles', 'et': 'Inversión en publicidad', 'u': 'S/ (soles)', 'mn': 0, 'mx': 1100, 'step': 10, 'tipo': 'num'},
    {'n': 'campana_escolar', 'et': '¿Hay campaña escolar?', 'u': '', 'mn': 0, 'mx': 1, 'step': 1, 'tipo': 'sel', 'op': ['Sin campaña', 'Con campaña']},
]


def predecir(valores):
    """Ecuación de la regresión: y = intercepto + suma(coef * x)."""
    y = MODELO["intercepto"]
    for nombre, coef in zip(MODELO["columnas"], MODELO["coeficientes"]):
        y += coef * valores[nombre]
    return max(y, 0)  # no tiene sentido un resultado negativo


@app.route("/", methods=["GET", "POST"])
def index():
    resultado, error, valores = None, None, {}
    if request.method == "POST":
        for c in CAMPOS:
            texto = request.form.get(c["n"], "").strip()
            valores[c["n"]] = texto
            try:
                v = float(texto)
            except ValueError:
                error = f"El campo «{c['et']}» debe ser un número."
                break
            # 3) Validación de rangos
            if not (c["mn"] <= v <= c["mx"]):
                error = f"«{c['et']}» debe estar entre {c['mn']} y {c['mx']} {c['u']}."
                break
            if c["tipo"] == "sel" and v not in (0, 1):
                error = f"«{c['et']}» solo admite 0 o 1."
                break
            valores[c["n"]] = v
        if error is None:
            y = predecir(valores)
            resultado = round(y)
    return render_template("index.html", campos=CAMPOS, valores=valores,
                           resultado=resultado, error=error,
                           titulo='Estimador de pedidos semanales de kits de robótica', unidad='pedidos de kits por semana', pie='Predicción con datos SIMULADOS. Es una estimación, no una garantía de ventas.',
                           metricas=MODELO["metricas"])


if __name__ == "__main__":
    app.run(debug=True)
```

**templates/index.html**
```html
<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>{{ titulo }}</title>
  <link rel="stylesheet" href="{{ url_for('static', filename='estilos.css') }}">
</head>
<body>
  <main class="tarjeta">
    <img class="logo" src="{{ url_for('static', filename='logo_senati.png') }}" alt="Logo SENATI">
    <h1>{{ titulo }}</h1>
    <p class="sub">Taller de Machine Learning y Deep Learning · Regresión lineal múltiple</p>

    <form method="post">
      {% for c in campos %}
      <label for="{{ c.n }}">{{ c.et }}{% if c.u %} <span class="unidad">({{ c.u }})</span>{% endif %}</label>
      {% if c.tipo == 'sel' %}
        <select id="{{ c.n }}" name="{{ c.n }}">
          {% for texto in c.op %}
          <option value="{{ loop.index0 }}" {% if valores.get(c.n)|string in [loop.index0|string, loop.index0|string ~ '.0'] %}selected{% endif %}>{{ texto }}</option>
          {% endfor %}
        </select>
      {% else %}
        <input id="{{ c.n }}" name="{{ c.n }}" type="number" min="{{ c.mn }}" max="{{ c.mx }}" step="{{ c.step }}"
               value="{{ valores.get(c.n, '') }}" placeholder="{{ c.mn }} a {{ c.mx }}" required>
      {% endif %}
      {% endfor %}
      <button type="submit">Calcular</button>
    </form>

    {% if error %}<div class="error">⚠ {{ error }}</div>{% endif %}
    {% if resultado is not none %}
    <div class="resultado">
      <span>Resultado estimado</span>
      <strong>{{ resultado }}</strong>
      <span>{{ unidad }}</span>
      <small>Error típico del modelo (MAE): ± {{ metricas.mae }}</small>
    </div>
    {% endif %}
    <p class="aviso">{{ pie }}</p>
  </main>
</body>
</html>
```

**static/estilos.css**
```css
* { box-sizing: border-box; }
body { margin: 0; font-family: "Segoe UI", Arial, sans-serif; background: #eef2ff; color: #1b1f3b; }
.tarjeta { max-width: 460px; margin: 30px auto; background: #fff; padding: 28px; border-radius: 14px; box-shadow: 0 6px 24px rgba(10,58,224,.15); }
.logo { height: 44px; }
h1 { font-size: 1.4rem; margin: 14px 0 4px; color: #0a3ae0; }
.sub { margin: 0 0 18px; color: #5b6180; font-size: .9rem; }
label { display: block; margin: 14px 0 6px; font-weight: 600; }
.unidad { font-weight: 400; color: #5b6180; }
input, select { width: 100%; padding: 10px; border: 1px solid #b9c2f0; border-radius: 8px; font-size: 1rem; }
button { width: 100%; margin-top: 20px; padding: 12px; border: 0; border-radius: 8px; background: #0a3ae0; color: #fff; font-size: 1.05rem; cursor: pointer; }
button:hover { background: #0830b8; }
.resultado { margin-top: 20px; padding: 16px; background: #e8f7ee; border-left: 5px solid #1e9e57; border-radius: 8px; text-align: center; }
.resultado strong { display: block; font-size: 2.2rem; color: #11703d; }
.resultado small { display: block; margin-top: 6px; color: #4a5a50; }
.error { margin-top: 18px; padding: 12px; background: #fdecec; border-left: 5px solid #d33; border-radius: 8px; }
.aviso { margin-top: 20px; font-size: .8rem; color: #6b7090; border-top: 1px solid #e1e5f7; padding-top: 12px; }
```

Para ejecutarla: `pip install flask scikit-learn pandas`, luego `python entrenar.py` y `python app.py`, y abrir http://127.0.0.1:5000

## Paso 4. Pruebas obligatorias
El mismo cálculo que hace la app (leyendo `modelo.json`):

In [4]:
import json

with open('modelo.json', encoding='utf-8') as f:
    M = json.load(f)

def predecir(valores):
    y = M['intercepto'] + sum(c * v for c, v in zip(M['coeficientes'], valores))
    return max(y, 0)

pruebas = [('Semana con 20 consultas, S/ 200 y sin campaña', [20, 200, 0]), ('Semana con 55 consultas, S/ 800 y con campaña', [55, 800, 1])]
for titulo, valores in pruebas:
    print(f'{titulo}  ->  {predecir(valores):.2f}')

Semana con 20 consultas, S/ 200 y sin campaña  ->  9.80
Semana con 55 consultas, S/ 800 y con campaña  ->  30.96


**Resultados:**

- Semana con 20 consultas, S/ 200 y sin campaña: **10**
- Semana con 55 consultas, S/ 800 y con campaña: **31**

**¿Tienen sentido?** Sí: el segundo caso da un valor mayor que el primero, y eso concuerda con los coeficientes positivos (más de cada insumo, mayor resultado). Ambos valores caen dentro del rango de los datos reales (3 a 41).

**¿En qué circunstancias podría fallar?** Puede fallar si: (1) se usan valores fuera de los rangos vistos (5–70 consultas, S/ 0–1100); (2) aparece un factor que el modelo no conoce (feriados, un competidor, vacaciones, precios); (3) la relación real no es lineal (más publicidad no siempre rinde igual); (4) cambia el comportamiento del mercado con el tiempo.

## Evidencias y evaluación
**Entregables:** notebook ejecutado, `datos.csv`, `entrenar.py`, `modelo.json`, aplicación Flask (`app.py`, `templates/index.html`, `static/estilos.css`), dos capturas de prueba y conclusiones.

### Explicación breve (10 líneas)
1. Se leyó `datos.csv` (110 filas, sin valores ausentes).
2. La respuesta `y` es `pedidos_kits`; las entradas `X` son consultas_web, inversion_publicidad_soles, campana_escolar.
3. Se separó 80 % para entrenar (88 filas) y 20 % para probar (22 filas).
4. Se entrenó una `LinearRegression` (regresión lineal múltiple).
5. Los coeficientes indican cuánto cambia `y` al subir 1 unidad cada variable.
6. Con los datos de prueba se obtuvo MAE = 2.95 pedidos y R² = 0.791.
7. Los coeficientes se guardaron en `modelo.json`.
8. `app.py` lee `modelo.json`, valida rangos y calcula la predicción al pulsar «Calcular».
9. La interfaz usa solo Flask, HTML y CSS, sin JavaScript.
10. Se probaron los dos casos obligatorios y se analizaron los límites del modelo.

### Conclusiones
- El modelo lineal explica una parte importante de la variación (R² = 0.79) y sirve como **estimación orientativa**, no como valor exacto.
- Las predicciones solo son confiables dentro de los rangos de entrada que vio el modelo.
- Los datos son **simulados**; con datos reales habría que reentrenar y volver a evaluar.
- Un siguiente paso sería probar otros modelos o agregar más variables y comparar el MAE.

### Rúbrica (20 pts)
Datos 3 · Modelo 5 · Métricas 4 · Aplicación 5 · Documentación 3